# Same sentence, different token counts

**Goal:** count the tokens in the *same* sentence written in English, Hindi, another natural language, code, and emoji — and see how much the token count changes.

Tokenizers: `tiktoken` encodings used by OpenAI models
- `cl100k_base` — GPT-4 / GPT-3.5
- `o200k_base` — GPT-4o and newer (bigger vocab, better multilingual coverage)

In [1]:
import tiktoken

encodings = {name: tiktoken.get_encoding(name) for name in ["cl100k_base", "o200k_base"]}

## The sentence in five forms

I don't have a native language, so the third one uses Japanese. Swap in your own native language in that slot.

In [2]:
sentences = {
    "English":  "The quick brown fox jumps over the lazy dog.",
    "Hindi":    "तेज़ भूरी लोमड़ी आलसी कुत्ते के ऊपर से कूदती है।",
    "Japanese": "素早い茶色の狐が怠け者の犬を飛び越える。",   # <- replace with your native language
    "Code":     'fox = Fox(speed="quick", color="brown")\nfox.jump_over(Dog(lazy=True))',
    "Emoji":    "⚡🟤🦊 ⤴️ 😴🐶",
}

## Count tokens

In [3]:
header = f"{'form':<10}{'chars':>7}{'bytes':>7}" + "".join(f"{n:>14}" for n in encodings)
print(header)
print("-" * len(header))
for label, text in sentences.items():
    counts = "".join(f"{len(enc.encode(text)):>14}" for enc in encodings.values())
    print(f"{label:<10}{len(text):>7}{len(text.encode('utf-8')):>7}{counts}")

form        chars  bytes   cl100k_base    o200k_base
----------------------------------------------------
English        44     44            10            10
Hindi          48    126            51            21
Japanese       20     60            30            20
Code           69     69            20            20
Emoji           9     27            18            16


## Relative cost vs. English (o200k_base)

In [4]:
enc = encodings["o200k_base"]
base = len(enc.encode(sentences["English"]))
for label, text in sentences.items():
    n = len(enc.encode(text))
    print(f"{label:<10}{n:>4} tokens  {n / base:>5.2f}x  " + "█" * n)

English     10 tokens   1.00x  ██████████
Hindi       21 tokens   2.10x  █████████████████████
Japanese    20 tokens   2.00x  ████████████████████
Code        20 tokens   2.00x  ████████████████████
Emoji       16 tokens   1.60x  ████████████████


## How each one gets split

In [5]:
for label, text in sentences.items():
    pieces = [enc.decode_single_token_bytes(t).decode("utf-8", errors="replace") for t in enc.encode(text)]
    print(f"{label} ({len(pieces)}):", pieces, "\n")

English (10): ['The', ' quick', ' brown', ' fox', ' jumps', ' over', ' the', ' lazy', ' dog', '.'] 

Hindi (21): ['ते', 'ज़', ' भ', 'ूरी', ' लो', 'म', 'ड़ी', ' आल', 'सी', ' क', 'ुत', '्त', 'े', ' के', ' ऊपर', ' से', ' क', 'ूद', 'ती', ' है', '।'] 

Japanese (20): ['素', '早', 'い', '茶', '色', 'の', '狐', 'が', '�', '�', 'け', '者', 'の', '犬', 'を', '飛', 'び', '越', 'える', '。'] 

Code (20): ['fox', ' =', ' Fox', '(speed', '="', 'quick', '",', ' color', '="', 'brown', '")\n', 'fox', '.jump', '_over', '(D', 'og', '(l', 'azy', '=True', '))'] 

Emoji (16): ['�', '�', '�', '�', '�', '�', '�', '�', ' �', '�', '�', '️', ' �', '�', '�', '�'] 



## Takeaways

- English is the cheapest: common words are usually one token each.
- Hindi (Devanagari) costs more — each character is 3 UTF-8 bytes and appears less in training data, so words get split into fragments. `o200k_base` narrows this gap a lot versus `cl100k_base`.
- Code tokenizes fairly efficiently, but `=`, quotes, brackets and underscores each add tokens.
- Emoji are 4 bytes each (plus invisible variation selectors), so a few emoji can cost as much as a whole sentence. `�` in the split view means one emoji was cut across several tokens.